In [73]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

train = pd.read_csv("../data/train.csv")
store = pd.read_csv("../data/store.csv")

C:\Users\Vikas\AppData\Local\Temp\ipykernel_20348\1847724447.py:5: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv("../data/train.csv")


In [74]:
df = train.merge(
    store,
    on="Store",
    how="left"
)

print(df.shape)

(1017209, 18)


In [75]:
df["Date"] = pd.to_datetime(df["Date"])

In [76]:
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"] = df["Date"].dt.day
df["Week"] = df["Date"].dt.isocalendar().week.astype(int)

In [77]:
df["DayOfYear"] = df["Date"].dt.dayofyear
df["Quarter"] = df["Date"].dt.quarter
df["IsWeekend"] = (
    df["DayOfWeek"] >= 6
).astype(int)

In [78]:
df["CompetitionDistance"] = (
    df["CompetitionDistance"]
    .fillna(df["CompetitionDistance"].median())
)

df["CompetitionOpenSinceMonth"] = (
    df["CompetitionOpenSinceMonth"]
    .fillna(0)
)

df["CompetitionOpenSinceYear"] = (
    df["CompetitionOpenSinceYear"]
    .fillna(0)
)

promo_cols = [
    "Promo2SinceWeek",
    "Promo2SinceYear"
]

for col in promo_cols:
    df[col] = df[col].fillna(0)


df["PromoAge"] = (
    df["Year"] - df["Promo2SinceYear"]
)

df["PromoAge"] = (
    df["PromoAge"]
    .clip(lower=0)
)

df["PromoInterval"] = (
    df["PromoInterval"]
    .fillna("None")
)

df["CompetitionAge"] = (
    df["Year"] - df["CompetitionOpenSinceYear"]
)

df["CompetitionAge"] = (
    df["CompetitionAge"]
    .clip(lower=0)
)

In [79]:
df.isnull().sum()

Store                        0
DayOfWeek                    0
Date                         0
Sales                        0
Customers                    0
Open                         0
Promo                        0
StateHoliday                 0
SchoolHoliday                0
StoreType                    0
Assortment                   0
CompetitionDistance          0
CompetitionOpenSinceMonth    0
CompetitionOpenSinceYear     0
Promo2                       0
Promo2SinceWeek              0
Promo2SinceYear              0
PromoInterval                0
Year                         0
Month                        0
Day                          0
Week                         0
DayOfYear                    0
Quarter                      0
IsWeekend                    0
PromoAge                     0
CompetitionAge               0
dtype: int64

In [80]:
df_model = df.drop(
    columns=[
        "Date",
        "Customers"
    ]
)

print(df_model.shape)

(1017209, 25)


In [81]:
cat_cols = [
    "StoreType",
    "Assortment",
    "StateHoliday",
    "PromoInterval"
]

df_model = pd.get_dummies(
    df_model,
    columns=cat_cols,
    drop_first=True
)

print(df_model.shape)

(1017209, 33)


In [82]:
X = df_model.drop("Sales", axis=1)
y = df_model["Sales"]

print(X.shape)
print(y.shape)

(1017209, 32)
(1017209,)


In [83]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(813767, 32)
(203442, 32)


# Model Training

### Linear Regression

In [86]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import (r2_score,mean_absolute_error,mean_squared_error)

In [87]:
lr = LinearRegression()

lr.fit(X_train, y_train)

LinearRegression()

In [88]:
preds = lr.predict(X_test)

In [89]:
r2 = r2_score(y_test, preds)
mae = mean_absolute_error(y_test,preds)
rmse = np.sqrt(mean_squared_error(y_test,preds))

print("R2:", r2)
print("MAE:", mae)
print("RMSE:", rmse)

R2: 0.5763530101815911
MAE: 1749.1416695096293
RMSE: 2503.0439431358786


### Random Forest

In [91]:
from sklearn.ensemble import RandomForestRegressor

In [92]:
rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

In [93]:
rf.fit(
    X_train,
    y_train
)

RandomForestRegressor(max_depth=15, n_jobs=-1, random_state=42)

In [113]:
preds = rf.predict(X_test)

In [115]:
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

import numpy as np

r2 = r2_score(y_test, preds)

mae = mean_absolute_error(
    y_test,
    preds
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        preds
    )
)

print("R²:", r2)
print("MAE:", mae)
print("RMSE:", rmse)

R²: 0.8375108594608819
MAE: 1014.1302400745359
RMSE: 1550.1673392193636


In [117]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print(feature_importance.head(10))

                      Feature  Importance
2                        Open    0.557270
3                       Promo    0.089028
5         CompetitionDistance    0.078049
0                       Store    0.059029
1                   DayOfWeek    0.028615
7    CompetitionOpenSinceYear    0.027816
6   CompetitionOpenSinceMonth    0.023623
15                  DayOfYear    0.020295
20                StoreType_b    0.017484
10            Promo2SinceYear    0.017461


### XGBoost

In [122]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb.fit(
    X_train,
    y_train
)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=8,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=300,
             n_jobs=-1, num_parallel_tree=None, ...)

In [123]:
preds = xgb.predict(X_test)

In [126]:
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

import numpy as np

r2 = r2_score(y_test, preds)

mae = mean_absolute_error(
    y_test,
    preds
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        preds
    )
)

print("R²:", r2)
print("MAE:", mae)
print("RMSE:", rmse)

R²: 0.9194750785827637
MAE: 744.8851410459508
RMSE: 1091.268597860657


In [128]:
import joblib

joblib.dump(
    xgb,
    "../models/sales_forecast_model.pkl"
)

['../models/sales_forecast_model.pkl']

In [130]:
joblib.dump(
    X.columns.tolist(),
    "../models/feature_columns.pkl"
)

['../models/feature_columns.pkl']